In [1]:
import os
import torch
import json
import math
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn.utils.rnn import pad_sequence
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version

# 加载 tokenier

In [5]:
model_dir = r"D:\learn-torch\Step6 Reinforcement Learnig\dpo\LLM-Research\Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)

# 设置pad token（llama3.2没有pad token）
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [6]:
device = torch.device("cuda")

# 策略模型
model = AutoModelForCausalLM.from_pretrained(model_dir)
model = model.to(device)

# 参考模型
ref_model = AutoModelForCausalLM.from_pretrained(model_dir)
ref_model.to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [7]:
dataset = load_dataset("parquet", data_files={
    'train': 'raw_data/data/train-00000-of-00001.parquet'})["train"]
print(f"Train size: {len(dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Train size: 5125


In [8]:
class PreferenceDataset(Dataset):
    """输出 (prompt, chosen_text, rejected_text)"""
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return item["prompt"], item["chosen"], item["rejected"]

In [9]:
def collate_fn(batch):
    input_ids, labels_list = [], []
    for prompt, chosen_resp, rejected_resp in batch:
        # tokenize prompt和 answer
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        chosen_ids = tokenizer(chosen_resp, add_special_tokens=False)["input_ids"]
        rejected_ids = tokenizer(rejected_resp, add_special_tokens=False)["input_ids"]

        # 把一条数据变2条，分别是choose和reject, [[prompt_ids, choose_id], [prompt_ids, rejected_ids]]
        input_ids += [
            torch.tensor(prompt_ids + chosen_ids, dtype = torch.long),
            torch.tensor(prompt_ids + rejected_ids, dtype = torch.long)
        ]
        # mask input token ids
        labels_list += [
            torch.tensor([-100]*len(prompt_ids) + chosen_ids, dtype = torch.long),
            torch.tensor([-100]*len(prompt_ids) + rejected_ids, dtype = torch.long)
        ]

    # batch做padding
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (input_ids != tokenizer.pad_token_id)
    labels_tensor = pad_sequence(labels_list, batch_first=True, padding_value=-100) # -100代表忽略loss计算

    return input_ids.to(device), attention_mask.to(device), labels_tensor.to(device)

In [10]:
batch_size = 4  # batch size
max_lr = 2e-6   # 学习率
min_lr = 1e-7   # 最小学习率
max_norm = 1.0  # 最大梯度范数
num_epochs = 1  # 训练epoch数
beta = 0.1      # DPO loss的beta系数
grad_acc_steps = 4  #梯度累积的步数

In [12]:
train_dataset = PreferenceDataset(dataset)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
)
max_steps = len(train_loader)
print(f"Total steps for one epoch: {max_steps}")

Total steps for one epoch: 1282


In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)
optimizer.zero_grad()

In [14]:
def get_lr(cur_step, max_steps, warmup_steps = None):
    warmup_steps = int(0.01*max_steps)
    # warmup
    if cur_step < warmup_steps:
        return max_lr * (cur_step+1) / warmup_steps
    # 超过max steps以最小学习率
    if cur_step > max_steps:
        return min_lr
    # 学习率线性decay
    ratio = (cur_step - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_lr + coeff * (max_lr - min_lr)

In [15]:
# DPO损失
def dpo_loss(nll_chosen, nll_rejected, ref_chosen, ref_rejected, beta):
    diff_theta = nll_rejected - nll_chosen #nll是 -log*pi，所以要取一个负号
    diff_ref = ref_rejected - ref_chosen
    loss = -F.logsigmoid(beta * (diff_theta - diff_ref)).mean()
    return loss

In [16]:
total_losses = []
for epoch in range(num_epochs):
    model.train()
    for step, (input_ids, attention_mask, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):

        # 设置模型梯度累积
        is_grad_accum_step = (step + 1) % grad_acc_steps == 0
        model.require_backward_grad_sync = is_grad_accum_step

        # 策略模型的logits, 用BF16精度训练
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(input_ids, attention_mask=attention_mask)
        logits  = outputs.logits  # [2*batch_size, seq_len, vocab_size]

        # 参考模型的logits
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                ref_outputs = ref_model(input_ids, attention_mask=attention_mask)
            ref_logits  = ref_outputs.logits # [2*batch_size, seq_len, vocab_size]

        # label左移一位
        logits = logits[...,:-1,:].contiguous() # [2*batch_size, seq_len-1, vocab_size]
        ref_logits = ref_logits[...,:-1,:].contiguous() # [2*batch_size, seq_len-1, vocab_size]
        labels = labels[...,1:].contiguous() # [2*batch_size, seq_len-1]

        # 计算损失
        vocab_size = logits.size(-1)
        bsz = logits.size(0)
        policy_loss  = F.cross_entropy(
            logits.view(-1, vocab_size), labels.view(-1), reduction="none").view(bsz, -1)
        ref_loss= F.cross_entropy(
            ref_logits.view(-1, vocab_size), labels.view(-1), reduction="none").view(bsz, -1)

        # loss求和
        policy_nll = policy_loss.sum(dim=1) # [batch_size]
        ref_nll = ref_loss.sum(dim=1) # [batch_size]

        # 把choose和reject的loss切分出来，[[prompt_ids, choose_id], [prompt_ids, rejected_ids]]
        nll_chosen = policy_nll[0::2]   # [batch_size]
        nll_rejected = policy_nll[1::2] # [batch_size]
        ref_chosen = ref_nll[0::2]
        ref_rejected = ref_nll[1::2]

        # 带入公式，计算 DPO loss
        loss = dpo_loss(nll_chosen, nll_rejected, ref_chosen, ref_rejected, beta)
        loss_value = loss.detach().item()
        total_losses.append(loss_value)

        # 打印日志
        if (step + 1) % 100 == 0:
            print(f'dpo loss at step {max_steps * epoch + step + 1} is: {loss_value:.4f}')

        # 反向传播梯度累积（这里不立即更新参数）
        loss = loss / grad_acc_steps # 这里要除一个累积的steps
        loss.backward()

        if is_grad_accum_step:
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
            # 调整学习率
            lr = get_lr(step, max_steps)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            # 参数更新
            optimizer.step()
            optimizer.zero_grad()

    # 保存 Checkpoint
    ckpt_dir = f"dpo_models/llama-3.2-1b-instruct-dpo-epoch{epoch+1}"
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)

Epoch 1:   0%|          | 6/1282 [00:44<2:38:52,  7.47s/it]

KeyboardInterrupt

